# Lecture 22: An embedding of extremals

**The challenge.** Lecture 21 generates exact extremals without solving
anything, but it cannot say where one ends: it chooses a start and
costates, and the end is wherever they lead. A robot asks the opposite
question, from where it is to where it must go, for any pair of states.
This lecture answers it with many generated extremals. Their endpoints are a
map of the space of motions, their costates are coordinates on that map,
and a query is answered by looking up a nearby extremal and polishing it
into the exact one between the requested states.

**After this lecture you can**

- say what is stored for each extremal and why its arcs matter as much as
  its costates;
- answer a query by nearest endpoints and a polish, and say what the
  polish's continuation and structure repair add;
- explain why the inverse map (endpoints to costates) is many-valued, and
  how a conditional normalizing flow represents that;
- describe a joint-embedding predictive architecture (JEPA) as a forward
  model in a learned space, and how inverting it proposes costates;
- read the results honestly: where the map works, and where it does not yet.

## 1. What is stored, and what a query does

Each generated extremal is stored with its start $x_0$, end $x_f$, time
$t_f$, costates $\lambda_0$, and its *structure*: the arcs it rides, in
order, with their junction values $(t_\text{in}, t_\text{out}, \pi)$. The
structure is the part a solve finds hardest (Lectures 15 and 20), and here
it is free: it came out of the generation.

A query $(x_0, x_f)$ then runs three steps, each a fallback for the one
before (`embedding.Polisher`):

1. **Polish.** Solve the boundary-value problem from $x_0$ to $x_f$ with
   `MultiArcShooting`, starting from a stored extremal's structure and values.
   Its own ends are close, so Newton starts close.
2. **Continue.** If that fails, solve a sequence of problems whose ends move
   in adaptive steps from the stored extremal's ends (where it is exact) to
   the query's. This works whenever the arcs need not change on the way.
3. **Repair.** If that fails too, regenerate from the query's own start with
   the stored costates and time, which finds the arcs that start leads to,
   and polish from there.

The nearest stored extremals are found by distance between endpoints, each
coordinate scaled by its spread (`embedding.EndpointIndex`). Symmetries
shrink the space: the axis below does not care where it is, only how far it
goes, so position enters as $x_f - x_0$. The six-joint arm has the same
symmetry in its first joint, which turns about gravity's axis.

Every answer passes the same checks as a generated extremal (no violation,
$\mu \ge 0$, $\pi \ge 0$, controls in their bounds). A poor proposal costs
time, never a wrong answer.

In [ ]:
import warnings

import numpy as np
import sympy as sp

from auto_optimal_control.embedding import EndpointIndex, ExtremalSet, Polisher
from auto_optimal_control.examples import jerk_limited_min_time
from auto_optimal_control.generation import ExtremalGenerator, Rejected
from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.problem import Box, OptimalControlProblem, PathConstraint

t, x, v, a, j = sp.symbols("t x v a j", real=True)
J, A = sp.symbols("J A", positive=True)
e0, ef = sp.symbols("x0 v0 a0", real=True), sp.symbols("xf vf af", real=True)
axis = OptimalControlProblem(
    name="axis_between", time=t, states=[x, v, a], controls=[j], params=[*e0, *ef, J, A],
    dynamics=[v, a, j], running_cost=1, initial_state=dict(zip([x, v, a], e0)),
    final_state=dict(zip([x, v, a], ef)), control_sets=[Box(j, -J, J)],
    path_constraints=[PathConstraint(a - A, "a_max"), PathConstraint(-a - A, "a_min")])

def params(x0, xf):
    return [*x0, *xf, 1.0, 1.0]

h = Homotopy(axis, policy="as_stated")
gen = ExtremalGenerator.from_homotopy(h, params(np.zeros(3), np.zeros(3)))
gen.max_seconds = 5.0
rng = np.random.default_rng(0)
stored = []
while len(stored) < 240:
    x0 = np.array([0.0, *rng.uniform(-0.5, 0.5, 2)]) if rng.random() < 0.5 else np.zeros(3)
    try:
        stored.append(gen.generate(x0, rng.normal(size=3), rng.uniform(0.5, 8), rng=rng))
    except Rejected:
        pass
data = ExtremalSet.from_extremals(stored)
print(len(data), "extremals;", sum(1 for s in data.arcs if s), "with arcs; most common structures:")
for s, n in data.structures()[:4]:
    print(f"  {n:4d}  {s}")

Now the queries a solve would be asked: rest to rest over a distance $D$.
None of them is in the set (no generated motion ends exactly at rest), and
the S-curve gives the answer in closed form.

In [ ]:
index = EndpointIndex(data, invariant=[0])
polish = Polisher(h, params, generator=gen)
print("   D    t_f (embedding)    t_f (closed form)   guesses  seconds  arcs")
for D in (0.5, 1.5, 3.0, 4.0, 6.0, 8.0):
    x0, xf = np.zeros(3), np.array([D, 0.0, 0.0])
    ids, _ = index.nearest(x0, xf, k=24)
    r = polish.solve(x0, xf, [index.shifted(i, x0) for i in ids])
    exact, _, _ = jerk_limited_min_time(1.0, 1.0, 1e9, D)
    print(f"{D:5.1f}  {r.tf:17.12f}  {exact:17.12f}  {r.tried:7d}  {r.seconds:7.2f}  {len(r.arcs) // 2}")

For short moves no limit is reached; from $D = 2$ on the motion rides
$a = A$ and then $a = -A$. In each case the two arcs came from stored
extremals, and nothing searched for them.

## 2. Why the inverse map needs more than a nearest neighbour

From a start, the map from costates (on $H = 0$) and time to the end is
smooth wherever the structure does not change. Its inverse, from the two
ends to the costates, has two difficulties that grow with the dimension.

- **It is many-valued.** Several extremals can join the same two states: a
  long way and a short way round, or two structures of equal cost. A
  regression from endpoints to costates averages the branches and lands on
  neither.
- **It has seams.** Where the structure changes, the costates move
  continuously but the arcs do not, and an answer near a seam needs the
  right side's arcs.

A nearest neighbour handles both for free (it returns real extremals, each
on one branch, with its arcs) but needs the set to be dense near the query,
and density costs exponentially in the dimension. Two kinds of learned
proposal aim at the same answer with less data. Each only proposes
$(\lambda_0, t_f)$; the generator turns a proposal into an extremal with its
arcs, from the query's start, and the polish does the rest. Both are trained
on generated indirect extremals and nothing else.

### A conditional normalizing flow

A normalizing flow is an invertible map $z = f_\theta(y; c)$ from the
targets $y = (\lambda_0/|\lambda_0|, \log t_f)$ to a standard normal $z$,
conditioned on the endpoints $c$. Its density follows from the change of
variables,

$$
\log p_\theta(y \mid c) = \log \mathcal{N}(f_\theta(y; c)) + \log \Big|\det \frac{\partial f_\theta}{\partial y}\Big|,
$$

and training maximizes it on the stored extremals. Built from affine
*couplings* (half the coordinates scale and shift the other half, with a
permutation between layers), the determinant is a sum of the scales, and
the inverse is as cheap as the forward map. Sampling $z$ and inverting gives
many proposals for one query, one per branch it has learned, ranked by
their density. That is exactly what a many-valued inverse needs.

### A joint-embedding predictive architecture

A JEPA learns to predict, in a learned embedding space, the representation
of one view of the data from another. Here the views are the two halves of
an extremal: a *context* encoder embeds the start and costates, a *target*
encoder embeds the end, and a predictor maps the first embedding onto the
second. The target encoder in the loss is an exponential moving average of
an online one, behind a stop-gradient, and a variance-covariance term (as in
VICReg) keeps the embeddings from collapsing to a constant.

It is a forward model, but not of the end itself: only of what the target
encoder keeps of it. The point of predicting in a learned space is that the
model need not reproduce what is hard to predict and does not matter for
choosing. To propose costates for a query, embed the goal, start from any
proposals (nearest neighbours here) and move their costates by gradient
descent until the predicted embedding matches the goal's. The model is
differentiable everywhere, including across the seams where the exact
forward map has kinks.

The code is in `auto_optimal_control.learn` (it needs PyTorch, the `learn`
extra), and the cell below shows how it is used. It is not run when these
notes are built.

In [ ]:
from auto_optimal_control import learn

C, Y = learn.training_data(data, index)               # conditions: endpoints; targets: (lam0 dir, log t_f)
flow, sc, sy = learn.train_flow(C, Y, epochs=400)

S = index.features(data.x0, data.x0)[:, :2]           # start: (v0, a0); position does not matter
E = index.features(data.x0, data.xf)[:, 2:]           # end: (xf - x0, vf, af)
jepa, ss, sy2, se = learn.train_jepa(S, Y, E, epochs=400)

def from_proposals(x0, ys):
    for y in ys:
        lam, T = learn.costates_of(y)
        try:
            yield gen.generate(x0, lam, T)            # its arcs, from the query's start
        except Rejected:
            pass

x0, xf = np.zeros(3), np.array([4.0, 0.0, 0.0])
by_flow = polish.solve(x0, xf, from_proposals(x0, learn.flow_proposals(flow, sc, sy, index.features(x0, xf)[0], 8)))
ids, _ = index.nearest(x0, xf, k=8)
ys, _ = learn.jepa_refine(jepa, ss, sy2, se, index.features(x0, x0)[0, :2], index.features(x0, xf)[0, 2:], Y[ids])
by_jepa = polish.solve(x0, xf, from_proposals(x0, ys))

### What they did

On the axis, with 900 generated extremals for training (364 with arcs) and
80 queries (60 held-out extremals, whose answers are known, and 20 rest to
rest moves, against the closed form), each method proposing 8 starting
points:

| proposals | solved, polish only | solved, with continuation and repair | median time per query |
|---|---|---|---|
| nearest endpoints | 72 / 80 | 78 / 80 | 0.05 s |
| conditional flow | 59 / 80 | 64 / 80 | 0.6 s to 1.3 s |
| JEPA, refining the nearest | 60 / 80 | 73 / 80 | 0.3 s |

Every accepted answer had the reference $t_f$: nothing wrong was accepted.
The flow trained in 13 s and the JEPA in 17 s on one core.

On a problem this small the nearest neighbour wins: three states leave the
endpoints dense with 900 extremals, and a neighbour hands over a real
extremal with its arcs, where a learned proposal has to be regenerated and
can land on the other side of a seam. The learned models earn their place
where the endpoints cannot be dense, which is the six-joint arm's 36
dimensions, and only once the generator can supply them with enough data
there. That is the next section's problem.

## 3. Six joints

The same pipeline on the six-joint arm in minimum time, with fixed limits
(jerks $\le 30$, torques set to 1.5 times the largest gravity torque plus a
margin), generated from rest at random poses by 15 processes for 45 minutes:

| | |
|---|---|
| extremals generated | 193 (about 3.5 minutes per extremal per core) |
| with arcs | 24, all on joint 1's torque limits |
| ending before the drawn time (truncated) | all 193 |
| rest-to-rest queries | 15, random poses |
| solved from the embedding | **0 / 15** (median 92 s spent trying) |
| solved by the automatic homotopy (Lecture 20) | 10 / 15 (median 80 s) |
| nearest stored endpoints, scaled distance | 2.3 to 10.2 (median 5.3) |

This is a negative result, and its cause is the generator, not the
embedding. Almost every draw from rest meets joint 1's torque limit within
a fraction of a second, with the jerks saturated, and most births fail for
the two reasons of Lecture 21, Section 4: the limit cannot be ridden with
the jerks in their bounds, or the saturated trajectory does not move with
the costates. Every draw is then cut short before its first unresolved
violation. So the set is small, its motions are short, and no stored end
is near a query's: across 36 endpoint dimensions, 193 extremals cannot be
dense, and a polish from five standard deviations away does not converge.

What would change this, roughly in order:

- **births that move the start** (`vary_start`), which removes the
  saturation problem at the cost of moving starts, and mixing in moving
  starts deliberately, since the queries want them anyway;
- **anchoring at a junction and integrating both ways** from it: choose a
  state on the limit that already satisfies the entry conditions, then
  integrate backward to a start and forward to an end. There the entry is
  exact by construction, which is where the backward direction really helps;
- **growing the set from its own answers**: every accepted query is a new
  stored extremal, and continuation from stored ones fills the space between;
- **then** the learned proposals, which reduce how dense the set must be.


## 4. Summary

- A set of generated extremals stores what a solve finds hardest, the
  structure, along with the costates. Querying it turns "from here to there"
  into a polish from a nearby exact extremal.
- Continuation in the endpoints and regeneration from the query's start
  cover what a nearest neighbour misses; every answer is checked like a
  generated extremal.
- The inverse map is many-valued and has seams. A conditional flow
  represents the branches; a JEPA is a differentiable forward model in a
  learned space, inverted by gradient descent. Both only propose.
- What decides how far this goes is the generator's yield of extremals
  with arcs in high dimension. The learned layers can reduce how dense the
  data must be, but not how much of it there is.